# 2η Εργασία – Μέρος Α: Full-State LSTM Forecasting
### 5-DOF Mass-Spring-Damper με Αρμονική Διέγερση

**Τι κάνουμε:** Εκπαιδεύουμε ένα LSTM να προβλέπει την επόμενη κατάσταση
`x_{n+1} = [u1..u5, v1..v5]` από τα τελευταία `p` βήματα του συστήματος.

**Πώς να τρέξεις:** `Shift+Enter` σε κάθε cell από πάνω προς τα κάτω.


## 1. Imports & Device

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import time, os

# ── Αναπαραγωγιμότητα: ίδια αποτελέσματα σε κάθε τρέξιμο ──────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── Device: MPS = Apple Silicon GPU, αλλιώς CPU ────────────────────────────
# Το M5 Mac έχει ενσωματωμένη GPU μέσω MPS (Metal Performance Shaders).
# Αν το MPS είναι διαθέσιμο, η εκπαίδευση γίνεται ~5-10x πιο γρήγορα!
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"✓ Χρησιμοποιούμε: {device}")
print(f"✓ PyTorch version: {torch.__version__}")


## 2. Φόρτωση & Οργάνωση Δεδομένων

Το CSV έχει 75 τροχιές × 1001 βήματα = 75.075 γραμμές.
Κάθε γραμμή = μία χρονική στιγμή μιας τροχιάς.


In [ ]:
# Αλλαξε το path αν το dataset είναι αλλού
DATA_PATH = "dataset.csv"

df = pd.read_csv(DATA_PATH)

# Τα columns του state x_n = [u1..u5, v1..v5]
STATE_COLS = ["u1","u2","u3","u4","u5","v1","v2","v3","v4","v5"]
FORCE_COL  = "F_top"   # F0*sin(Omega*t) – εξωτερική διέγερση

# Διαχωρισμός με βάση τη στήλη 'split' (ήδη ορισμένη στο CSV)
train_df = df[df["split"] == "train"]
test_df  = df[df["split"] == "test"]

train_ids = sorted(train_df["traj_id"].unique())
test_ids  = sorted(test_df["traj_id"].unique())

print(f"✓ Training τροχιές : {len(train_ids)}")
print(f"✓ Test τροχιές     : {len(test_ids)}")
print(f"✓ Βήματα/τροχιά    : {df[df['traj_id']==0].shape[0]}")
print(f"\nΠρώτες 3 γραμμές:")
df.head(3)


In [ ]:
def extract_trajectories(df_sub, ids):
    """
    Επιστρέφει list από (X, F) tuples:
      X : (1001, 10) numpy array – full state [u1..u5, v1..v5]
      F : (1001,)    numpy array – external force F_top
    """
    trajs = []
    for tid in ids:
        traj = df_sub[df_sub["traj_id"] == tid].sort_values("t")
        X = traj[STATE_COLS].to_numpy(dtype=np.float32)
        F = traj[FORCE_COL].to_numpy(dtype=np.float32)
        trajs.append((X, F))
    return trajs

train_trajs = extract_trajectories(train_df, train_ids)
test_trajs  = extract_trajectories(test_df,  test_ids)

# Γρήγορος έλεγχος
X0, F0 = train_trajs[0]
print(f"✓ Shape ενός X: {X0.shape}  (Nt × 10 state)")
print(f"✓ Shape ενός F: {F0.shape}  (Nt,)")
print(f"✓ u5 range: [{X0[:,4].min():.4f}, {X0[:,4].max():.4f}]")
print(f"✓ F_top range: [{F0.min():.4f}, {F0.max():.4f}]")


## 3. Κανονικοποίηση (Normalization)

**Γιατί κανονικοποιούμε;**

Τα νευρωνικά δίκτυα εκπαιδεύονται πολύ καλύτερα όταν τα inputs έχουν παρόμοια
κλίμακα (mean≈0, std≈1). Εδώ τα `u` (μετατοπίσεις, ~10⁻³) και `v` (ταχύτητες, ~10⁻¹)
διαφέρουν πολύ σε μέγεθος — χωρίς normalization το δίκτυο θα αγνοούσε τα μικρά u.

**Κανόνας:** Υπολογίζουμε mean/std **μόνο από το training set** και εφαρμόζουμε
στο test set. Αν χρησιμοποιούσαμε και τα test data θα «διέρρεε» πληροφορία.


In [ ]:
# Stack όλα τα training snapshots για υπολογισμό στατιστικών
all_X_train = np.vstack([X for X, F in train_trajs])    # (60×1001, 10)
all_F_train = np.concatenate([F for X, F in train_trajs])  # (60×1001,)

X_mean = all_X_train.mean(axis=0).astype(np.float32)   # (10,)
X_std  = all_X_train.std(axis=0).astype(np.float32)    # (10,)
F_mean = float(all_F_train.mean())
F_std  = max(float(all_F_train.std()), 1e-8)

# Αποφυγή διαίρεσης με 0
X_std = np.where(X_std < 1e-8, np.float32(1.0), X_std)

print("State means (≈0 λόγω μηδενικών αρχικών συνθηκών):")
print(dict(zip(STATE_COLS, X_mean.round(5))))
print("\nState stds (βλέπουμε ότι u<<v σε κλίμακα):")
print(dict(zip(STATE_COLS, X_std.round(5))))

def normalize(X, F):
    """Κανονικοποίηση με mean/std του training set."""
    return (X - X_mean) / X_std, (F - F_mean) / F_std

def denormalize(X_norm):
    """Αντίστροφη κανονικοποίηση για visualization."""
    return X_norm * X_std + X_mean


## 4. Κατασκευή Dataset – Sliding Window

**Η λογική:** Από κάθε τροχιά 1001 βημάτων, με παράθυρο `p=20` παράγουμε
981 samples. Κάθε sample:
- **Input:**  `(p, 11)` tensor — p βήματα × [10 state + 1 force]
- **Target:** `(10,)` tensor — το επόμενο state

```
γραμμές [0→19]  → είσοδος,  γραμμή [20]  → target  (sample 0)
γραμμές [1→20]  → είσοδος,  γραμμή [21]  → target  (sample 1)
...
γραμμές [980→999] → είσοδος, γραμμή [1000] → target (sample 980)
```


In [ ]:
WINDOW = 20  # p: μήκος παραθύρου (= 0.40 sec)

def build_tensors(trajs):
    """
    Δημιουργεί (X_tensor, Y_tensor) από λίστα τροχιών.
    
    Αντί για lazy loading, φτιάχνουμε τα tensors από την αρχή:
    - Γρηγορότερο κατά την εκπαίδευση
    - 68MB RAM συνολικά – απολύτως διαχειρίσιμο
    """
    X_list, Y_list = [], []
    
    for X, F in trajs:
        Xn, Fn = normalize(X, F)
        # Συνένωση: (1001, 10) + (1001, 1) → (1001, 11)
        arr = np.column_stack([Xn, Fn]).astype(np.float32)
        
        # Sliding window
        for start in range(len(arr) - WINDOW):
            end = start + WINDOW
            X_list.append(arr[start:end])    # (p, 11)
            Y_list.append(arr[end, :10])     # (10,) – μόνο state, όχι force
    
    X_tensor = torch.tensor(np.array(X_list))   # (N, p, 11)
    Y_tensor = torch.tensor(np.array(Y_list))   # (N, 10)
    return X_tensor, Y_tensor

print("Φτιάχνω tensors... ", end="", flush=True)
t0 = time.time()
X_train, Y_train = build_tensors(train_trajs)
X_test,  Y_test  = build_tensors(test_trajs)
print(f"έτοιμο σε {time.time()-t0:.1f}s")

print(f"\nTraining tensors: X={X_train.shape}, Y={Y_train.shape}")
print(f"Test tensors:     X={X_test.shape},  Y={Y_test.shape}")
mem_mb = (X_train.numel() + Y_train.numel() + X_test.numel() + Y_test.numel()) * 4 / 1e6
print(f"Μνήμη: {mem_mb:.0f} MB")

# DataLoaders: χωρίζουν τα data σε mini-batches για την εκπαίδευση
# shuffle=True στο train: τυχαία σειρά samples σε κάθε epoch
BATCH_SIZE = 1024
train_loader = DataLoader(TensorDataset(X_train, Y_train),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test, Y_test),
                          batch_size=BATCH_SIZE, shuffle=False)

print(f"\nBatch size: {BATCH_SIZE}")
print(f"Batches/epoch (train): {len(train_loader)}")


## 5. Αρχιτεκτονική LSTM

```
Input (B, p, 11)
      ↓
  LSTM layer(s)    ← εδώ γίνεται η "μαγεία" – τα gates θυμούνται/ξεχνούν
      ↓
 τελευταίο timestep output (B, hidden_size)
      ↓
  Linear layer     ← απλό y = Wx + b
      ↓
Output (B, 10)     ← πρόβλεψη x_{n+1}
```

**Παράμετροι LSTM:**
- `input_size=11`: διάσταση κάθε timestep
- `hidden_size`: πόσες «νευρώνες» κρυφής κατάστασης
- `num_layers`: πόσα LSTM στρώματα στοιβαγμένα
- `batch_first=True`: tensors έχουν μορφή (batch, seq, features)


In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=11, hidden_size=128, 
                 num_layers=2, output_size=10, dropout=0.1):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            # dropout εφαρμόζεται ΜΕΤΑΞΥ layers (regularization)
            dropout     = dropout if num_layers > 1 else 0.0
        )
        
        # Μικρό MLP στο τέλος: hidden → hidden/2 → output
        # Δίνει περισσότερη εκφραστικότητα από ένα απλό Linear
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, output_size)
        )
    
    def forward(self, x):
        # x: (batch, seq_len=p, input_size=11)
        lstm_out, _ = self.lstm(x)
        # lstm_out: (batch, seq_len, hidden_size)
        # Κρατάμε μόνο το τελευταίο timestep
        last = lstm_out[:, -1, :]    # (batch, hidden_size)
        return self.head(last)       # (batch, 10)


# Δημιουργία μοντέλου και μεταφορά στη σωστή συσκευή (MPS/CPU)
HIDDEN_SIZE = 128
NUM_LAYERS  = 2
model = LSTMForecaster(
    input_size  = 11,
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    output_size = 10,
    dropout     = 0.1
).to(device)

# Μέτρηση παραμέτρων
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Μοντέλο στο {device}")
print(f"✓ Συνολικές εκπαιδεύσιμες παράμετροι: {n_params:,}")
print()
print(model)


## 6. Εκπαίδευση

**Loss function:** MSE (Mean Squared Error) — μετράει πόσο μακριά είναι η πρόβλεψη
από την πραγματική τιμή: `loss = mean((ŷ - y)²)`

**Optimizer:** Adam — ο πιο δημοφιλής για deep learning. Προσαρμόζει αυτόματα
το learning rate ανά παράμετρο.

**Gradient clipping:** αποτρέπει το πρόβλημα των «exploding gradients» που
εμφανίζεται συχνά σε RNN/LSTM.

**Learning rate scheduler:** μειώνει το lr αν το test loss σταματήσει να βελτιώνεται.


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=5)

EPOCHS = 60

train_losses = []
test_losses  = []
best_test_loss = float('inf')
best_model_state = None

print(f"Εκπαίδευση για {EPOCHS} epochs...")
print(f"{'Epoch':>5} | {'Train MSE':>10} | {'Test MSE':>10} | {'LR':>8} | {'Time':>6}")
print("-" * 52)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    # ── Training ──────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        # Μεταφορά batch στη συσκευή (MPS/CPU)
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()          # Μηδένισμα gradients
        y_pred = model(X_batch)        # Forward pass
        loss   = criterion(y_pred, y_batch)   # Υπολογισμός loss
        loss.backward()                # Backward pass (autograd)
        
        # Clipping: κόβει τα gradients αν ξεπεράσουν norm=1.0
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()               # Ενημέρωση βαρών
        train_loss += loss.item() * len(X_batch)
    
    train_loss /= len(X_train)
    
    # ── Evaluation ────────────────────────────────────────────────────────
    model.eval()
    test_loss = 0.0
    with torch.no_grad():    # Δεν χρειαζόμαστε gradients για evaluation
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred  = model(X_batch)
            test_loss += criterion(y_pred, y_batch).item() * len(X_batch)
    
    test_loss /= len(X_test)
    scheduler.step(test_loss)
    
    # Αποθήκευση καλύτερου μοντέλου
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_model_state = {k: v.cpu().clone() 
                            for k, v in model.state_dict().items()}
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    
    lr_now = optimizer.param_groups[0]['lr']
    if epoch % 10 == 0 or epoch == 1:
        print(f"{epoch:>5} | {train_loss:>10.5f} | {test_loss:>10.5f} | "
              f"{lr_now:>8.1e} | {time.time()-t0:>5.1f}s")

# Φόρτωση καλύτερου μοντέλου
model.load_state_dict(best_model_state)
model.eval()
print(f"\n✓ Εκπαίδευση ολοκληρώθηκε. Καλύτερο test MSE: {best_test_loss:.5f}")


In [ ]:
# ── Γράφημα καμπυλών εκπαίδευσης ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(train_losses, 'b-', lw=2, label='Train MSE')
ax.semilogy(test_losses,  'r-', lw=2, label='Test MSE')
ax.set(xlabel='Epoch', ylabel='MSE (log scale)',
       title='Καμπύλες Εκπαίδευσης – LSTM (h=128, L=2, p=20)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 7. Αξιολόγηση – One-Step Ahead Prediction

Σε κάθε βήμα n, δίνουμε στο LSTM τα **πραγματικά** p προηγούμενα states
(από το dataset) και ζητάμε μόνο το επόμενο. Το σφάλμα δεν αθροίζεται.


In [ ]:
def predict_one_step(X_full, F_full):
    """
    X_full: (1001, 10) unnormalized
    F_full: (1001,)    unnormalized
    Returns: (1001-p, 10) predictions in original scale
    """
    Xn, Fn = normalize(X_full, F_full)
    arr = np.column_stack([Xn, Fn]).astype(np.float32)
    
    # Φτιάχνουμε όλα τα input windows ταυτόχρονα (vectorized, γρήγορο)
    windows = np.array([arr[s:s+WINDOW] for s in range(len(arr)-WINDOW)])
    
    with torch.no_grad():
        inp   = torch.tensor(windows).to(device)
        preds = model(inp).cpu().numpy()
    
    return denormalize(preds)   # πίσω στην αρχική κλίμακα


# Αξιολόγηση σε όλες τις test τροχιές
print("One-step RMSE ανά DOF (μέσος 15 test τροχιών):")
print(f"{'DOF':<6} | {'RMSE':>10}")
print("-" * 20)

rmse_1step = np.zeros((len(test_trajs), 10))
for i, (X, F) in enumerate(test_trajs):
    preds   = predict_one_step(X, F)
    targets = X[WINDOW:]
    rmse_1step[i] = np.sqrt(np.mean((preds - targets)**2, axis=0))

for j, col in enumerate(STATE_COLS):
    print(f"{col:<6} | {rmse_1step[:,j].mean():>10.3e}")

print(f"\n→ Μέσο RMSE: {rmse_1step.mean():.3e}")


## 8. Αξιολόγηση – Autoregressive Rollout

Τώρα το μοντέλο τροφοδοτείται με τις **δικές του προβλέψεις**:

1. Seed: τα πρώτα `p` βήματα από το dataset (πραγματικά)
2. Βήμα 1: πρόβλεψη `x_{p+1}` → μπαίνει στο buffer
3. Βήμα 2: πρόβλεψη `x_{p+2}` χρησιμοποιώντας την πρόβλεψη ως input
4. κ.ο.κ.

Κάθε σφάλμα «μολύνει» την επόμενη πρόβλεψη — αναμένουμε αύξηση σφάλματος με τον χρόνο.

> Σημείωση: Για τη δύναμη F_top χρησιμοποιούμε πάντα τις **πραγματικές** τιμές
> (γιατί είναι ντετερμινιστική — F₀·sin(Ω·t) — δεν εξαρτάται από το state).


In [ ]:
def predict_autoregressive(X_full, F_full):
    """
    X_full: (1001, 10) unnormalized
    F_full: (1001,)    unnormalized
    Returns: (1001-p, 10) predictions in original scale
    """
    Xn, Fn = normalize(X_full, F_full)
    Fn32   = Fn.astype(np.float32)
    
    # Buffer: αρχικοποιείται με τα πρώτα p πραγματικά states
    buf   = Xn[:WINDOW].astype(np.float32).copy()  # (p, 10)
    preds = []
    
    with torch.no_grad():
        for step in range(len(Xn) - WINDOW):
            # Παράθυρο δύναμης: πάντα πραγματικές τιμές
            F_window = Fn32[step : step+WINDOW].reshape(-1, 1)  # (p, 1)
            
            # Input: [buffer states | force window] → (p, 11)
            inp = np.concatenate([buf, F_window], axis=1)
            inp_t = torch.tensor(inp).unsqueeze(0).to(device)  # (1, p, 11)
            
            # Πρόβλεψη επόμενου state
            pred = model(inp_t).cpu().numpy()[0]   # (10,)
            preds.append(pred)
            
            # Ενημέρωση buffer: διώχνουμε το παλαιότερο, προσθέτουμε την πρόβλεψη
            buf[:-1] = buf[1:]
            buf[-1]  = pred
    
    return denormalize(np.array(preds))


# Αξιολόγηση
print("Autoregressive RMSE ανά DOF (μέσος 15 test τροχιών):")
print(f"{'DOF':<6} | {'RMSE 1-step':>12} | {'RMSE Auto':>12} | {'Λόγος':>8}")
print("-" * 45)

rmse_auto = np.zeros((len(test_trajs), 10))
for i, (X, F) in enumerate(test_trajs):
    preds   = predict_autoregressive(X, F)
    targets = X[WINDOW:]
    rmse_auto[i] = np.sqrt(np.mean((preds - targets)**2, axis=0))

for j, col in enumerate(STATE_COLS):
    r1 = rmse_1step[:,j].mean()
    ra = rmse_auto[:,j].mean()
    print(f"{col:<6} | {r1:>12.3e} | {ra:>12.3e} | {ra/r1:>8.1f}x")

print(f"\n→ Μέσο RMSE auto: {rmse_auto.mean():.3e}  "
      f"({rmse_auto.mean()/rmse_1step.mean():.1f}x χειρότερο από one-step)")


## 9. Γραφήματα Προβλέψεων

In [ ]:
# Επιλέγουμε μία test τροχιά για visualization
TRAJ_IDX = 0
X_vis, F_vis = test_trajs[TRAJ_IDX]
t_axis = np.arange(1001) * 0.02    # χρονικός άξονας σε seconds
t_pred = t_axis[WINDOW:]           # από το p-οστό βήμα και μετά

p1_vis  = predict_one_step(X_vis, F_vis)
par_vis = predict_autoregressive(X_vis, F_vis)
tgt_vis = X_vis[WINDOW:]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Προβλέψεις LSTM – Test Trajectory {TRAJ_IDX}  '
             f'(F₀={test_df[test_df.traj_id==test_ids[TRAJ_IDX]].F0.iloc[0]:.2f}, '
             f'Ω={test_df[test_df.traj_id==test_ids[TRAJ_IDX]].Omega.iloc[0]:.2f} rad/s)',
             fontsize=12)

plot_cfg = [
    (0, 'u1 – Κόμβος 1 (βάση)'),
    (4, 'u5 – Κόμβος 5 (κορυφή, σημείο διέγερσης)'),
    (5, 'v1 – Ταχύτητα κόμβου 1'),
    (9, 'v5 – Ταχύτητα κόμβου 5'),
]

for ax, (dof, lbl) in zip(axes.flat, plot_cfg):
    ax.plot(t_pred, tgt_vis[:,dof],  'k-',  lw=1.8, label='Ground truth', zorder=3)
    ax.plot(t_pred, p1_vis[:,dof],   'b--', lw=1.3, label='One-step',     zorder=4)
    ax.plot(t_pred, par_vis[:,dof],  'r-',  lw=1.0, label='Autoregressive', alpha=0.85, zorder=2)
    ax.set(xlabel='t (s)', title=lbl)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 10. Συσσώρευση Σφάλματος στο Χρόνο

In [ ]:
# Υπολογισμός RMSE σε κάθε χρονικό βήμα για όλες τις test τροχιές
all_err_1step = []
all_err_auto  = []

for X, F in test_trajs:
    p1  = predict_one_step(X, F)
    par = predict_autoregressive(X, F)
    tgt = X[WINDOW:]
    # RMSE ανά timestep (μέσος πάνω από όλα τα DOFs)
    all_err_1step.append(np.sqrt(np.mean((p1  - tgt)**2, axis=1)))
    all_err_auto.append( np.sqrt(np.mean((par - tgt)**2, axis=1)))

all_err_1step = np.array(all_err_1step)  # (15, 981)
all_err_auto  = np.array(all_err_auto)

m1, s1 = all_err_1step.mean(0), all_err_1step.std(0)
ma, sa = all_err_auto.mean(0),  all_err_auto.std(0)
t_err  = t_axis[WINDOW:]

fig, ax = plt.subplots(figsize=(11, 5))
ax.semilogy(t_err, m1, 'b-', lw=2, label='One-step (mean)')
ax.fill_between(t_err, np.clip(m1-s1, 1e-10, None), m1+s1,
                alpha=0.15, color='blue', label='One-step ±1σ')
ax.semilogy(t_err, ma, 'r-', lw=2, label='Autoregressive (mean)')
ax.fill_between(t_err, np.clip(ma-sa, 1e-10, None), ma+sa,
                alpha=0.15, color='red', label='Autoregressive ±1σ')
ax.set(xlabel='t (s)', ylabel='RMSE (log scale)',
       title='Συσσώρευση Σφάλματος – 15 Test Trajectories')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\nRMSE στο t=0.4s (αρχή):")
print(f"  One-step:      {m1[0]:.3e}")
print(f"  Autoregressive:{ma[0]:.3e}")
print(f"\nRMSE στο t=20s (τέλος):")
print(f"  One-step:      {m1[-1]:.3e}")
print(f"  Autoregressive:{ma[-1]:.3e}")
print(f"  → Αύξηση auto: {ma[-1]/ma[0]:.1f}x από την αρχή")


## 11. Ερώτημα 5 – Sensitivity Analysis Hyperparameters

Δοκιμάζουμε γρήγορα (15 epochs) διαφορετικές επιλογές για:
- **(i)** Window size p
- **(ii)** Hidden size / num layers  
- **(iii)** Normalization on/off


In [ ]:
def quick_train(hidden=128, layers=2, window=20, use_norm=True, epochs=15, lr=1e-3):
    """Γρήγορη εκπαίδευση για σύγκριση hyperparameters."""
    def build(trajs, w):
        Xs, Ys = [], []
        for X, F in trajs:
            if use_norm:
                Xn, Fn = normalize(X, F)
            else:
                Xn, Fn = X, F   # χωρίς normalization
            arr = np.column_stack([Xn, Fn]).astype(np.float32)
            for s in range(len(arr)-w):
                Xs.append(arr[s:s+w]); Ys.append(arr[s+w,:10])
        return torch.tensor(np.array(Xs)), torch.tensor(np.array(Ys))
    
    Xt, Yt = build(train_trajs, window)
    Xe, Ye = build(test_trajs,  window)
    tdl = DataLoader(TensorDataset(Xt, Yt), batch_size=1024, shuffle=True)
    edl = DataLoader(TensorDataset(Xe, Ye), batch_size=1024)
    
    m   = LSTMForecaster(11, hidden, layers, 10, 0.0).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    crit = nn.MSELoss()
    
    for _ in range(epochs):
        m.train()
        for Xb, yb in tdl:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); l = crit(m(Xb), yb)
            l.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
    
    m.eval(); tel = 0
    with torch.no_grad():
        for Xb, yb in edl:
            Xb, yb = Xb.to(device), yb.to(device)
            tel += crit(m(Xb), yb).item() * len(Xb)
    return tel / len(Xe)

# (i) Window size
print("(i) Window size p  [hidden=128, layers=2, 15 epochs]")
ws_results = {}
for ws in [5, 10, 20, 40]:
    l = quick_train(window=ws)
    ws_results[ws] = l
    print(f"  p={ws:3d} → test MSE = {l:.4f}")

# (ii) Architecture
print("\n(ii) Architecture  [p=20, 15 epochs]")
arch_results = {}
for h, la in [(64,1), (128,1), (128,2), (256,2)]:
    l = quick_train(hidden=h, layers=la)
    arch_results[(h,la)] = l
    print(f"  hidden={h}, layers={la} → test MSE = {l:.4f}")

# (iii) Normalization
print("\n(iii) Normalization  [p=20, hidden=128, layers=2, 15 epochs]")
l_norm   = quick_train(use_norm=True,  lr=1e-3)
l_nonorm = quick_train(use_norm=False, lr=1e-4)  # μικρότερο lr χωρίς norm
print(f"  Με normalization   : {l_norm:.4f}")
print(f"  Χωρίς normalization: {l_nonorm:.4f}")


In [ ]:
# Γραφήματα sensitivity
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Hyperparameter Sensitivity (15 epochs, test MSE)', fontsize=12)

# (i)
axes[0].bar([str(k) for k in ws_results], list(ws_results.values()), color='steelblue', alpha=0.85)
axes[0].set(xlabel='Window size p', ylabel='Test MSE', title='(i) Window size')
axes[0].grid(alpha=0.3, axis='y')

# (ii)
labels = [f'h={h}\nL={l}' for h,l in arch_results]
axes[1].bar(labels, list(arch_results.values()), color='coral', alpha=0.85)
axes[1].set(xlabel='Αρχιτεκτονική', ylabel='Test MSE', title='(ii) Hidden & Layers')
axes[1].grid(alpha=0.3, axis='y')

# (iii)
axes[2].bar(['Με\nnorm.', 'Χωρίς\nnorm.'], [l_norm, l_nonorm],
            color=['steelblue', 'tomato'], alpha=0.85)
axes[2].set(ylabel='Test MSE', title='(iii) Normalization')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.show()


## 12. Αποθήκευση Μοντέλου (για χρήση στο Μέρος Β)

In [ ]:
# Αποθηκεύουμε το μοντέλο ΚΑΙ τα normalization stats
# Χρειαζόμαστε τα stats για να κάνουμε σωστό denormalize στο Μέρος Β
checkpoint = {
    'model_state_dict': model.state_dict(),
    'model_config': {
        'hidden_size': HIDDEN_SIZE,
        'num_layers':  NUM_LAYERS,
        'window':      WINDOW,
    },
    'norm_stats': {
        'X_mean': X_mean,
        'X_std':  X_std,
        'F_mean': F_mean,
        'F_std':  F_std,
    },
    'train_losses': train_losses,
    'test_losses':  test_losses,
}

torch.save(checkpoint, 'lstm_part_a.pth')
print("✓ Αποθηκεύτηκε: lstm_part_a.pth")
print("  → Θα χρησιμοποιηθεί στο Μέρος Β (PCA + LSTM)")

# Σύνοψη
print("\n" + "="*50)
print("ΣΥΝΟΨΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ – ΜΕΡΟΣ Α")
print("="*50)
print(f"Μοντέλο    : LSTM (h={HIDDEN_SIZE}, L={NUM_LAYERS}, p={WINDOW})")
print(f"Best MSE   : {best_test_loss:.5f} (normalized space)")
print(f"\n{'DOF':<6} | {'RMSE 1-step':>12} | {'RMSE Auto':>12}")
print("-"*35)
for j, col in enumerate(STATE_COLS):
    print(f"{col:<6} | {rmse_1step[:,j].mean():>12.3e} | {rmse_auto[:,j].mean():>12.3e}")
